# 청킹 기준 탐색 노트북

**탐색용입니다.** 여기서 정한 기준을 확정하면 `.py`로 옮기고, 이 노트북은 확인용으로만 남깁니다.
커밋 전에 모든 셀 출력을 지웁니다(PDF 원문이 출력에 남지 않게).

확인할 질문
1. 쪽 하나가 얼마나 긴가? (쪽 단위 청크로 충분한가)
2. 긴 쪽 안에 제목을 찾을 수 있는가? (제목 기준으로 쪼갤 수 있는가)
3. 어느 방식이 청크 크기를 고르게 하는가?

In [ ]:
import sys, warnings, collections
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
for extra in (ROOT / "src", ROOT / "src" / "card_benefit"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import pandas as pd
import pymupdf

INDEX = ROOT / "src" / "card_benefit" / "data" / "index"
DATA = ROOT / "src" / "data"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

def target_docs(issuer):
    # 대표 상품설명서 중 텍스트 PDF, 신용카드만
    d = pd.read_csv(INDEX / issuer / "documents.csv", encoding="utf-8-sig", dtype=str).fillna("")
    d = d[(d.canonical == "Y") & (d.doc_role == "product_guide") & (d.ocr_needed == "N") & (d.card_type == "credit")]
    return d.assign(issuer=issuer)

docs = pd.concat([target_docs("hana"), target_docs("woori")], ignore_index=True)
print("대상 문서", docs.groupby("issuer").size().to_dict())

## 1. 쪽 하나의 길이 분포

In [ ]:
rows = []
for r in docs.itertuples():
    for no, page in enumerate(pymupdf.open(DATA / r.file_path), start=1):
        rows.append({"issuer": r.issuer, "doc": r.file_path.rsplit("/", 1)[1][:40], "file_path": r.file_path,
                     "page": no, "chars": len(page.get_text().strip())})
pages = pd.DataFrame(rows)
print("전체 쪽", len(pages))
print(pages.groupby("issuer").chars.describe(percentiles=[.1, .5, .9, .99]).round(0).to_string())
print()
print("글자 없는 쪽:", (pages.chars == 0).sum(), "/ 200자 미만:", (pages.chars < 200).sum(), "/ 4000자 초과:", (pages.chars > 4000).sum())

In [ ]:
# 구간별 쪽 수 (한눈에 분포 보기)
bins = [-1, 0, 200, 1000, 2000, 3000, 4000, 6000, 10**6]
labels = ["0", "1~200", "201~1000", "1001~2000", "2001~3000", "3001~4000", "4001~6000", "6001~"]
pages["bucket"] = pd.cut(pages.chars, bins=bins, labels=labels)
pages.pivot_table(index="bucket", columns="issuer", values="page", aggfunc="count", fill_value=0, observed=False)

In [ ]:
# 가장 긴 쪽 10개 (쪼갤 필요가 있는지 보기)
pages.sort_values("chars", ascending=False).head(10)[["issuer", "doc", "page", "chars"]]

## 2. 제목을 찾을 수 있나? (글자 크기·굵기)

PDF는 글자마다 크기와 굵기 정보가 있습니다. 본문 크기보다 크거나 굵고 짧은 줄을 제목 후보로 봅니다.

In [ ]:
def page_lines(page):
    # 줄마다 (글, 가장 큰 글자 크기, 굵은 글자 비율)을 만든다
    lines = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = [s for s in line["spans"] if s["text"].strip()]
            if not spans:
                continue
            text = "".join(s["text"] for s in spans).strip()
            size = max(s["size"] for s in spans)
            bold = sum(len(s["text"]) for s in spans if s["flags"] & 16 or "Bold" in s["font"]) / max(1, sum(len(s["text"]) for s in spans))
            lines.append((text, round(size, 1), round(bold, 2)))
    return lines

def body_size(lines):
    # 글자 수가 가장 많은 크기를 본문 크기로 본다
    count = collections.Counter()
    for text, size, _ in lines:
        count[size] += len(text)
    return count.most_common(1)[0][0] if count else 0

def show_lines(file_path, page_no, limit=40):
    lines = page_lines(pymupdf.open(DATA / file_path)[page_no - 1])
    base = body_size(lines)
    print("본문 크기", base)
    for text, size, bold in lines[:limit]:
        mark = "◆" if size >= base + 1.5 or (bold >= 0.8 and len(text) <= 30 and size >= base) else " "
        print(f"{mark} {size:5} {bold:4}  {text[:60]}")

sample = docs[docs.file_path.str.contains("102862")].iloc[0].file_path
show_lines(sample, 1)

## 3. 두 방식 비교 (초안 규칙)

- **A. 쪽 단위**: 쪽 하나가 청크 하나 (지금 `traceability`가 하는 방식)
- **B. 제목 단위**: 쪽 안에서 제목 후보가 나올 때마다 새 청크 (글자가 너무 적은 조각은 앞 조각에 붙임)

In [ ]:
MIN_CHARS = 400   # 이보다 짧은 조각은 앞 조각에 붙인다 (기준값 비교 후 150에서 400으로)

def is_heading(text, size, bold, base):
    # 굵기 규칙을 더해도 결과가 거의 같아서 글자 크기만 쓴다
    return size >= base + 1.5

def split_by_heading(page):
    lines = page_lines(page)
    base = body_size(lines)
    chunks, current = [], []
    for text, size, bold in lines:
        if is_heading(text, size, bold, base) and sum(len(t) for t in current) >= MIN_CHARS:
            chunks.append("\n".join(current))
            current = []
        current.append(text)
    if current:
        chunks.append("\n".join(current))
    return chunks

def chunk_sizes(file_path):
    a, b = [], []
    for page in pymupdf.open(DATA / file_path):
        text = page.get_text().strip()
        if not text:
            continue
        a.append(len(text))
        b.extend(len(c) for c in split_by_heading(page))
    return a, b

rows = []
for r in docs.itertuples():
    a, b = chunk_sizes(r.file_path)
    rows += [{"issuer": r.issuer, "방식": "A 쪽 단위", "chars": n} for n in a]
    rows += [{"issuer": r.issuer, "방식": "B 제목 단위", "chars": n} for n in b]
cmp = pd.DataFrame(rows)
cmp.groupby(["issuer", "방식"]).chars.describe(percentiles=[.1, .5, .9]).round(0)

In [ ]:
# B 방식 청크가 어떻게 나뉘는지 눈으로 확인 (샘플 한 쪽)
page = pymupdf.open(DATA / sample)[1]
for i, chunk in enumerate(split_by_heading(page), start=1):
    print(f"--- 청크 {i} ({len(chunk)}자) ---")
    print(chunk[:200].replace("\n", " | "))

## 3-2. C. 혼합 방식

- 쪽이 `SPLIT_OVER`자 이하이면 **쪽 하나가 청크 하나**(A와 같음)
- 그보다 긴 쪽만 제목 기준으로 쪼갠다(B와 같은 규칙, 조각이 `MIN_CHARS`보다 짧으면 앞 조각에 붙임)

In [ ]:
SPLIT_OVER = 4000   # 기준값 비교 후 3000에서 4000으로

def hybrid_chunks(page):
    text = page.get_text().strip()
    if not text:
        return []
    if len(text) <= SPLIT_OVER:
        return [text]
    return split_by_heading(page)

rows = []
for r in docs.itertuples():
    a, b = chunk_sizes(r.file_path)
    c = [len(x) for page in pymupdf.open(DATA / r.file_path) for x in hybrid_chunks(page)]
    rows += [{"issuer": r.issuer, "방식": "C 혼합", "chars": n} for n in c]
cmp_all = pd.concat([cmp, pd.DataFrame(rows)], ignore_index=True)
summary = cmp_all.groupby(["issuer", "방식"]).chars.describe(percentiles=[.1, .5, .9]).round(0)
summary["4000자 초과"] = cmp_all[cmp_all.chars > 4000].groupby(["issuer", "방식"]).size()
summary["200자 미만"] = cmp_all[cmp_all.chars < 200].groupby(["issuer", "방식"]).size()
summary.fillna(0).astype({"4000자 초과": int, "200자 미만": int})

## 4. 판단 메모

- **쪽 길이 분포**: 중앙값 약 1,300자, 4,000자 넘는 쪽 12%(98개), 최대 9,539자. 글자 없는 쪽 37개.
- **제목 후보**: 글자 크기만으로도 찾을 수 있다(우리 샘플 본문 6.0pt, 제목 8.4pt). 굵기 규칙을 더해도 결과가 거의 같아서 **크기만 쓴다**.
- **방식**: B(제목 단위)는 조각이 너무 작아지고, A(쪽 단위)는 긴 쪽이 남는다. **C(혼합)** 를 쓴다.
- **기준값 비교**(대표 상품설명서 193개, `SPLIT_OVER`=쪽 길이 기준, `MIN_CHARS`=최소 조각 길이):

| 발급사 | SPLIT_OVER | MIN_CHARS | 청크 수 | 중앙값 | 4000자 초과 | 200자 미만 |
|---|---|---|---|---|---|---|
| hana | 3000 | 150 | 793 | 328 | 13 | 286 |
| hana | 4000 | 400 | 485 | 814 | 13 | 55 |
| woori | 3000 | 150 | 785 | 879 | 8 | 116 |
| woori | 4000 | 400 | 506 | 1,196 | 8 | 45 |

- **선택**: `SPLIT_OVER`=4000, `MIN_CHARS`=400, 제목은 글자 크기(본문+1.5pt 이상)만 사용.
- **남은 문제**: 4000자를 넘는 청크가 하나 21개(하나 13, 우리 8) 남는다. 제목이 없는 긴 쪽이라 제목으로는 못 쪼갠다. 200자 미만 조각 100개는 대부분 원래 짧은 쪽(표지, 안내 문구)이라 그대로 둔다.